# Employee Attrition Dataset — Feature Engineering Pipeline

**Subject:** Feature Engineering
**Business Context:** HR Analytics team at a multinational technology organization is preparing a machine-learning-ready dataset to study employee attrition, ahead of future predictive modelling work.

**Notebook Structure**
- Part A — Understanding the Dataset (EDA)
- Part B — Feature Engineering Strategy (missing values, transformations, encoding)
- Part C — Feature Construction & Feature Selection
- Part D — Dimensionality Reduction & Reusable Pipeline
- Part E — Summary & Reflection



In [0]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Display settings for better readability
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 100

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries loaded successfully.")


In [0]:
# Load the dataset
DATA_PATH = "/Workspace/Users/kushagrakumar1205@gmail.com/bits_msc_ds_ai/Sem_1/Feature Engineering/Graded Assignment Dataset.csv"
df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()   # Working copy; df_raw is kept untouched as a reference
print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(df.head().to_string())

---
## Part A — Understanding the Dataset

### A.1 Structure, Feature Types & Summary


In [0]:
print("COLUMN DATA TYPES")
print()
print(df.dtypes)
print()

print("FEATURE TYPE CLASSIFICATION (business definition, per data dictionary)")
print()

feature_types = {
    'Identifier':   ['Employee_ID'],
    'Numeric (continuous/discrete)': [
        'Age','Total_Work_Experience','Years_at_Company','Years_in_Current_Role',
        'Promotion_Count','Distance_From_Home','Monthly_Salary','Annual_Bonus',
        'Stock_Option_Value','Overtime_Hours','Monthly_Expenses','Training_Hours',
        'Certification_Count','Project_Count','Engagement_Index','Productivity_Index',
        'Financial_Stress_Index','Attendance_Score','Career_Growth_Index','Annual_Compensation'
    ],
    'Ordinal (encoded as numeric 1-10 scales)': [
        'Performance_Rating','Manager_Rating','Job_Satisfaction','Work_Life_Balance',
        'Environment_Satisfaction','Relationship_Satisfaction','Recognition_Score','Manager_Support'
    ],
    'Ordinal (categorical labels)': ['Education_Level','Compensation_Band','Experience_Band'],
    'Nominal (categorical)': ['Gender','Marital_Status','Department','Job_Role','Business_Travel','Work_Mode'],
    'Target (binary)': ['Attrition']
}
for k, v in feature_types.items():
    print(f"\n{k} ({len(v)}):")
    print(" ", v)

In [0]:
print("Numeric summary statistics:")
print()
num_cols_all = df.select_dtypes(include=[np.number]).columns.tolist()
summary = df[num_cols_all].describe().T
summary['skew'] = df[num_cols_all].skew()
print(summary.round(2).to_string())

### A.2 Missing Values

Missing values were identified in four columns, all in the 5–7% range — consistent with real-world enterly/system-integration gaps rather than a systemic collection failure.


In [0]:
# Calculate missing value counts and percentages
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_table = pd.DataFrame({'Missing_Count': missing, 'Missing_%': missing_pct})
missing_table['Missing_%'] = missing_table['Missing_%'].apply(lambda x: f'{x}%')
print(missing_table.to_string())
print()

# Visualize missing values with bar chart
plt.figure(figsize=(7,4))
ax = sns.barplot(x=missing_table.index, y=missing_table['Missing_Count'], color='#4C72B0')
plt.xlabel('')
plt.ylabel('% Missing')
plt.title('Missing Value Percentage by Feature')
plt.xticks(rotation=20)

for i, (idx, row) in enumerate(missing_table.iterrows()):
    ax.text(i, row['Missing_Count'], row['Missing_Count'], 
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [0]:
# Check if missingness is related to attrition (MAR vs MCAR analysis)
# If attrition rates are similar, data is likely Missing At Random (MAR)
for col in missing_table.index:
    flag = df[col].isnull().astype(int)
    rate_missing = df.loc[flag==1, 'Attrition'].mean()
    rate_present = df.loc[flag==0, 'Attrition'].mean()
    print(f"{col:28s} | Attrition rate when MISSING: {rate_missing:.3f}  | when PRESENT: {rate_present:.3f}")

**Observation:** Attrition rates are similar whether these values are missing or present, suggesting the missingness is **Missing At Random (MAR)** / unrelated to the outcome — i.e. likely due to reporting gaps (e.g., a manager rating not yet submitted, training not yet logged) rather than employees systematically withholding information. This supports using standard imputation (median/mode) rather than treating "missing" itself as a signal.


### A.3 Duplicates & Identifier Check

In [0]:
print("Duplicate Employee_ID:", df['Employee_ID'].duplicated().sum())
print("Fully duplicated rows:", df.duplicated().sum())
print("Unique Employee_IDs:", df['Employee_ID'].nunique(), "out of", len(df), "rows -> confirms one row per employee, no duplication issues.")

### A.4 Numerical Distributions — Skew & Shape

In [0]:
skewed_cols = ['Monthly_Salary','Annual_Bonus','Stock_Option_Value','Annual_Compensation',
               'Distance_From_Home','Overtime_Hours','Years_in_Current_Role','Years_at_Company',
               'Monthly_Expenses','Certification_Count']

fig, axes = plt.subplots(2, 5, figsize=(20, 7))
axes = axes.flatten()
for i, col in enumerate(skewed_cols):
    sns.histplot(df[col].dropna(), kde=True, ax=axes[i], color='#55A868')
    axes[i].set_title(f"{col}\nskew={df[col].skew():.2f}")
plt.tight_layout()
plt.suptitle("Right-Skewed Numeric Features", y=1.03, fontsize=14)
plt.show()

**Observation:** Compensation-related features (`Monthly_Salary`, `Annual_Bonus`, `Stock_Option_Value`, `Annual_Compensation`) and tenure-based counts (`Years_in_Current_Role`, `Years_at_Company`, `Certification_Count`) show strong right skew (skew > 1, several > 2). This is typical of monetary and count data, where most employees cluster at lower values with a long tail of senior/highly-compensated employees. These will need log-style transformation before use in distance-based or linear models.


### A.5 Outlier Detection (IQR Method)

In [0]:
# Detect outliers using IQR method (1.5 * IQR rule)
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols.remove('Attrition')

outlier_summary = []
for c in num_cols:
    Q1, Q3 = df[c].quantile(0.25), df[c].quantile(0.75)
    IQR = Q3 - Q1
    lb, ub = Q1 - 1.5*IQR, Q3 + 1.5*IQR  # Lower and upper bounds
    n_out = ((df[c] < lb) | (df[c] > ub)).sum()
    if n_out > 0:
        outlier_summary.append((c, n_out, round(n_out/len(df)*100, 2)))

outlier_df = pd.DataFrame(outlier_summary, columns=['Feature','Outlier_Count','Outlier_%']).sort_values('Outlier_%', ascending=False)
print(outlier_df.to_string(index=False))

In [0]:
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
axes = axes.flatten()
top_outlier_cols = outlier_df['Feature'].head(8).tolist()
for i, col in enumerate(top_outlier_cols):
    sns.boxplot(x=df[col], ax=axes[i], color='#DD8452')
    axes[i].set_title(col)
plt.tight_layout()
plt.suptitle("Boxplots — Features with Highest Outlier Rates", y=1.03, fontsize=14)
plt.show()

**Observation:** Outlier rates are moderate (1–6.5%) and concentrated in compensation, benefits, and tenure fields. These are **plausible extreme values** (e.g., a senior employee with very high stock options) rather than data entry errors — none of the max values are physically impossible (e.g., no negative ages, no salaries below minimum wage). We will treat these via **capping/winsorization** rather than removal, to avoid losing genuine senior-employee signal while limiting their leverage on scale-sensitive models.


### A.6 Correlation Analysis

In [0]:
plt.figure(figsize=(16, 13))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0, annot=False, linewidths=0.3, cbar_kws={'shrink':0.7})
plt.title("Correlation Heatmap — Numeric Features", fontsize=14)
plt.tight_layout()
plt.show()

In [0]:
# Identify strongly correlated feature pairs (|r| > 0.6) for redundancy analysis
high_corr_pairs = []
for i in range(len(num_cols)):
    for j in range(i+1, len(num_cols)):
        c = corr.iloc[i, j]
        if abs(c) > 0.6:
            high_corr_pairs.append((num_cols[i], num_cols[j], round(c, 3)))

high_corr_df = pd.DataFrame(high_corr_pairs, columns=['Feature_1','Feature_2','Correlation']).sort_values('Correlation', key=abs, ascending=False)
print(high_corr_df.to_string(index=False))

In [0]:
# Verify if Annual_Compensation is derived from other compensation features
# Check: Annual_Compensation = Monthly_Salary*12 + Annual_Bonus + Stock_Option_Value
check = df['Monthly_Salary']*12 + df['Annual_Bonus'].fillna(0) + df['Stock_Option_Value']
diff = (df['Annual_Compensation'] - check).abs()
print("Annual_Compensation vs (Monthly_Salary*12 + Annual_Bonus + Stock_Option_Value)")
print("Median absolute difference:", diff.median())
print("95th percentile difference:", diff.quantile(0.95))
print("-> Confirms Annual_Compensation is an ENGINEERED SUM of the three compensation components (near-zero residual).")

**Key correlation observations:**
1. **`Annual_Compensation` is algebraically derived** from `Monthly_Salary`, `Annual_Bonus`, and `Stock_Option_Value` (residual ≈ 0). Keeping all four creates perfect/near-perfect multicollinearity.
2. **`Age` and `Total_Work_Experience`** are highly correlated (r = 0.94) — both capture "career maturity."
3. **Engagement/survey items** (`Job_Satisfaction`, `Relationship_Satisfaction`, `Recognition_Score`, `Manager_Support`) are all moderately-to-strongly correlated with `Engagement_Index` (r = 0.75–0.79), suggesting `Engagement_Index` behaves like a composite of the individual survey scores.
4. **`Performance_Rating`, `Manager_Rating`, `Productivity_Index`** form another correlated cluster (r = 0.76–0.88).

### A.7 Categorical Feature Distributions

In [0]:
cat_cols = ['Gender','Marital_Status','Department','Job_Role','Business_Travel','Work_Mode','Education_Level','Compensation_Band','Experience_Band']
fig, axes = plt.subplots(3, 3, figsize=(17, 13))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    order = df[col].value_counts().index
    sns.countplot(x=df[col], order=order, ax=axes[i], color='#4C72B0')
    axes[i].set_title(col)
    axes[i].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.suptitle("Categorical Feature Distributions", y=1.02, fontsize=14)
plt.show()

### A.8 Target Variable — Class Balance

In [0]:
attr_counts = df['Attrition'].value_counts()
attr_pct = df['Attrition'].value_counts(normalize=True)*100
print("Attrition counts:\n", attr_counts.to_string())
print("\nAttrition %:\n", attr_pct.round(2).to_string())

plt.figure(figsize=(5,4))
sns.countplot(x=df['Attrition'], hue=df['Attrition'], palette=['#4C72B0','#C44E52'], legend=False)
plt.xticks([0,1], ['Stayed (0)','Left (1)'])
plt.title('Target Variable Distribution')
plt.show()

**Observation:** The dataset is **imbalanced** — about 19% attrition vs. 81% retention. Evaluation metrics like accuracy would be misleading, and techniques like class weighting or SMOTE should be considered at model-training time.


### A.9 Quick Bivariate View — Numeric Features vs Attrition

In [0]:
bivar_cols = ['Monthly_Salary','Overtime_Hours','Years_at_Company','Job_Satisfaction',
              'Distance_From_Home','Engagement_Index','Promotion_Count','Financial_Stress_Index']
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for i, col in enumerate(bivar_cols):
    sns.boxplot(x='Attrition', y=col, data=df, hue='Attrition', ax=axes[i],
                palette=['#4C72B0','#C44E52'], legend=False)
    axes[i].set_xticks([0,1])
    axes[i].set_xticklabels(['Stayed','Left'])
plt.tight_layout()
plt.suptitle("Numeric Features vs Attrition", y=1.03, fontsize=14)
plt.show()

### A.10 Summary of Key Observations (Part A)

| # | Observation | Implication for Feature Engineering |
|---|---|---|
| 1 | 4 columns have 5–7% missing values (MAR pattern) | Median/mode imputation is appropriate; add missing-indicator flags as candidate features |
| 2 | No duplicate rows or duplicate Employee_IDs | No deduplication needed |
| 3 | `Employee_ID` is a pure identifier | Drop before modelling |
| 4 | Compensation & tenure features are strongly right-skewed | Apply log1p transformation |
| 5 | Outliers present (1–6.5%) but plausible, not erroneous | Cap/winsorize rather than delete |
| 6 | `Annual_Compensation` is an exact derived sum of 3 other features | Drop to remove perfect multicollinearity |
| 7 | `Age` and `Total_Work_Experience` are highly correlated (r=0.94) | Resolve via VIF-based selection / engineered ratio in Part C |
| 8 | Survey/engagement scores form correlated clusters | Consider a composite index / PCA candidate |
| 9 | 3 ordinal categorical columns (`Education_Level`, `Compensation_Band`, `Experience_Band`) | Use explicit ordinal encoding (not one-hot) to preserve order |
| 10 | 6 nominal categorical columns | Use one-hot encoding |
| 11 | Target is imbalanced (~19% attrition) | Flag for downstream model-training decisions (not addressed in this FE pipeline) |


## Part B — Designing a Feature Engineering Strategy

Working principle for this section: every transformation is **fit-able as a reusable step**, and every decision has an explicit justification. We work on `df` (a copy of the raw data) from here on.


### B.1 Dropping Pure Identifiers
`Employee_ID` carries no predictive signal and is unique per row — it is dropped immediately (kept aside only as an index reference).

In [0]:
employee_ids = df['Employee_ID'].copy()
df = df.drop(columns=['Employee_ID'])
print("Dropped Employee_ID. Remaining columns:", df.shape[1])

### B.2 Handling Missing Values

Per the Part A analysis, missingness in `Training_Hours`, `Annual_Bonus`, `Manager_Rating`, `Environment_Satisfaction` appears to be MAR (unrelated to Attrition). Strategy:

- **Numeric, skewed** (`Training_Hours`, `Annual_Bonus`) → **median imputation** (robust to skew/outliers, unlike mean).
- **Ordinal survey/rating scores** (`Manager_Rating`, `Environment_Satisfaction`) → **median imputation** as well, since these are integer-like ordinal scales where the median is a valid, interpretable category (unlike mean, which could produce non-integer scores).
- We additionally create **binary "was-missing" indicator flags** for these 4 columns. Even though missingness looked non-informative on average, keeping the flag costs little and lets a downstream model detect any subtler pattern the summary stats missed. This is a defensible, low-risk addition rather than a strong claim of informativeness.

*Alternative considered:* KNN or regression-based imputation would exploit correlations between features (e.g., `Manager_Rating` correlates with `Performance_Rating`) and could be more accurate, but adds complexity and a dependency on other features. Given the modest missing rate (≤7%) and MAR pattern, simple median imputation was judged sufficiently effective and easier to reproduce in production.


In [0]:
cols_with_na = ['Training_Hours', 'Annual_Bonus', 'Manager_Rating', 'Environment_Satisfaction']

# Create missing-indicator flags before imputation (to preserve information about missingness pattern)
for col in cols_with_na:
    df[f'{col}_was_missing'] = df[col].isnull().astype(int)

# Perform median imputation (robust to skew and outliers)
impute_medians = {}
for col in cols_with_na:
    med = df[col].median()
    impute_medians[col] = med
    df[col] = df[col].fillna(med)

print("Imputation medians used:")
for k, v in impute_medians.items():
    print(f"  {k}: {v}")
print("\nRemaining nulls in dataset:", df.isnull().sum().sum())

### B.3 Outlier Treatment — Winsorization (Capping)

Rather than deleting outlier rows (which would discard genuine senior/high-earning employees and shrink an already-imbalanced attrition class), we **cap extreme values at the 1st and 99th percentiles** for the numeric features flagged with meaningful outlier rates in Part A. This limits the leverage of extreme points on scale-sensitive downstream models while preserving every row.

*Alternative considered:* Full IQR-based capping (`Q1 - 1.5*IQR`, `Q3 + 1.5*IQR`) is more aggressive and would cap 3–6% of values quite close to the bulk of the distribution for very skewed features like `Stock_Option_Value`; percentile-based (1st/99th) capping was chosen as a gentler, distribution-agnostic alternative that only trims the most extreme 2% of values in total.


In [0]:
# Cap outliers at 1st and 99th percentiles (winsorization)
cap_cols = ['Stock_Option_Value','Annual_Bonus','Distance_From_Home','Overtime_Hours',
            'Monthly_Expenses','Monthly_Salary','Years_in_Current_Role','Years_at_Company',
            'Certification_Count']

cap_bounds = {}
for col in cap_cols:
    lo, hi = df[col].quantile(0.01), df[col].quantile(0.99)
    cap_bounds[col] = (lo, hi)
    df[col] = df[col].clip(lower=lo, upper=hi)  # Clip extreme values to bounds

print("Capping bounds (1st / 99th percentile) applied:")
for k, (lo, hi) in cap_bounds.items():
    print(f"  {k:24s}: [{lo:.2f}, {hi:.2f}]")

### B.4 Transforming Skewed Numeric Features

Highly right-skewed features (|skew| > 1 after capping) are transformed with `log1p` (log(1+x), safe for zero values) to reduce skew and make distributions closer to symmetric — this benefits linear models and distance-based methods (e.g. KNN, regularized regression, PCA) that assume roughly comparable, well-behaved feature scales.

Tree-based models are insensitive to monotonic transforms like log, so this step primarily helps the *linear/distance-based* branch of a future modelling pipeline — but it does no harm to tree-based methods either, so it's a safe default.


In [0]:
# Identify columns with high skew after capping (|skew| > 1)
skew_after_cap = df[cap_cols + ['Annual_Compensation']].skew().sort_values(ascending=False)
print("Skew after capping:")
print(skew_after_cap.round(2).to_string())

log_transform_cols = skew_after_cap[skew_after_cap.abs() > 1].index.tolist()
print("\nColumns selected for log1p transform (|skew| > 1):", log_transform_cols)

In [0]:
for col in log_transform_cols:
    df[f'{col}_log'] = np.log1p(df[col])

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
axes = axes.flatten()
for i, col in enumerate(log_transform_cols[:6]):
    sns.histplot(df[f'{col}_log'], kde=True, ax=axes[i], color='#8172B2')
    axes[i].set_title(f"{col}_log\nskew={df[f'{col}_log'].skew():.2f}")
plt.tight_layout()
plt.suptitle("Log-Transformed Features (post-capping)", y=1.03)
plt.show()

print("\nNote: original (untransformed) columns are retained for now; the final feature")
print("selection in Part C decides whether to keep the raw, the log version, or both.")

### B.5 Encoding Categorical Variables

Two distinct encoding strategies, matched to variable type:

1. **Ordinal categoricals** (`Education_Level`, `Compensation_Band`, `Experience_Band`) → **explicit ordinal integer encoding** using a manually defined order (not alphabetical, and not one-hot), so the natural rank (e.g., High School < Bachelor < Master < Doctorate) is preserved as usable numeric information.
2. **Nominal categoricals** (`Gender`, `Marital_Status`, `Department`, `Job_Role`, `Business_Travel`, `Work_Mode`) → **one-hot encoding**, since there is no inherent order and one-hot avoids imposing a false rank that label/ordinal encoding would introduce.

*Alternative considered:* Target/mean encoding was considered for high-cardinality nominal columns, but with only 3–7 categories per nominal column here, one-hot encoding is simple, avoids target-leakage risk, and doesn't need cross-validation-safe encoding schemes.


In [0]:
# Apply ordinal encoding to categorical variables with natural ordering
ordinal_maps = {
    'Education_Level':  {'High School': 0, 'Bachelor': 1, 'Master': 2, 'Doctorate': 3},
    'Compensation_Band': {'Low': 0, 'Medium': 1, 'High': 2},
    'Experience_Band':   {'Entry': 0, 'Junior': 1, 'Mid': 2, 'Senior': 3},
}

for col, mapping in ordinal_maps.items():
    print(f"{col} unique values: {sorted(df[col].unique())}")
    df[col + '_enc'] = df[col].map(mapping)

print("\nSample of encoded ordinal columns:")
print(df[['Education_Level','Education_Level_enc','Compensation_Band','Compensation_Band_enc',
          'Experience_Band','Experience_Band_enc']].head().to_string())

# Drop the original string columns now that the ordinal-encoded numeric versions exist —
# keeping both would be pure duplication (identical information, two representations).
df = df.drop(columns=list(ordinal_maps.keys()))
print("\nDropped raw ordinal string columns (superseded by *_enc):", list(ordinal_maps.keys()))

In [0]:
# Apply one-hot encoding to nominal categorical variables (no natural order)
nominal_cols = ['Gender','Marital_Status','Department','Job_Role','Business_Travel','Work_Mode']
df_encoded = pd.get_dummies(df, columns=nominal_cols, drop_first=True)  # Drop first to avoid multicollinearity
print(f"Shape before one-hot encoding: {df.shape}")
print(f"Shape after one-hot encoding:  {df_encoded.shape}")
new_ohe_cols = [c for c in df_encoded.columns if c not in df.columns]
print(f"\nNew one-hot columns created ({len(new_ohe_cols)}):")
print(new_ohe_cols)
df = df_encoded

### B.6 Scaling Numeric Features

Scaling is deferred to the **pipeline stage (Part D)** rather than applied destructively here, because the correct scaler must be **fit only on training data** in a real ML workflow to avoid data leakage. We use `StandardScaler` (zero mean, unit variance) as the default choice for the final pipeline since:
- Most engineered numeric features are roughly continuous rather than strictly bounded.
- `StandardScaler` is the standard, safe default for linear models, PCA, and distance-based methods used downstream.

*Alternative considered:* `MinMaxScaler` would be preferable if a downstream model specifically required bounded [0,1] inputs (e.g., some neural network activation ranges); `RobustScaler` (median/IQR-based) was also considered given the outliers, but since we already winsorized extreme values in B.3, `StandardScaler` on the capped data is sufficient and keeps the pipeline simpler.


In [0]:
print('Part B core steps complete. Current shape:', df.shape)

## Part C — Feature Construction & Feature Selection


### C.1 Constructing New Features

Each new feature is designed around a concrete HR business hypothesis about attrition drivers, using ratios and composites that are more directly interpretable than raw counts.


In [0]:
# 1. Tenure_Ratio: how much of an employee's total career has been spent at THIS company
#    Business intuition: a low ratio for an experienced person suggests they are relatively new
#    here despite a long career -> may not yet feel "anchored", a potential attrition risk factor.
df['Tenure_Ratio'] = df['Years_at_Company'] / (df['Total_Work_Experience'] + 1)

# 2. Promotion_Rate: promotions earned per year at the company
#    Business intuition: slow promotion velocity relative to tenure is a classic driver of attrition
#    (career stagnation).
df['Promotion_Rate'] = df['Promotion_Count'] / (df['Years_at_Company'] + 1)

# 3. Role_Stability: fraction of company tenure spent in the CURRENT role
#    Business intuition: an employee stuck a long time in the same role (low turnover into new
#    roles) may feel less growth -> career-stagnation proxy, distinct from Promotion_Rate.
df['Role_Stability'] = df['Years_in_Current_Role'] / (df['Years_at_Company'] + 1)

# 4. Income_per_Experience: salary "value" relative to how experienced the employee is
#    Business intuition: an experienced employee earning relatively little for their experience
#    level is more likely to feel under-compensated and look elsewhere.
df['Income_per_Experience'] = df['Monthly_Salary'] / (df['Total_Work_Experience'] + 1)

# 5. Overtime_Intensity: overtime hours normalised by workload (project count)
#    Business intuition: high overtime given a LOW project count suggests inefficiency/overload
#    on individual tasks rather than just "busy people work overtime" -> burnout risk signal.
df['Overtime_Intensity'] = df['Overtime_Hours'] / (df['Project_Count'] + 1)

# 6. Wellbeing_Composite: average of 4 correlated satisfaction/relationship survey scores
#    Business intuition: consolidates several correlated survey items (see Part A.6) into a
#    single, more robust "employee wellbeing" signal while reducing redundancy.
wellbeing_items = ['Job_Satisfaction','Work_Life_Balance','Relationship_Satisfaction','Recognition_Score']
df['Wellbeing_Composite'] = df[wellbeing_items].mean(axis=1)

# 7. Age_at_Joining: implied age when the employee joined the company
#    Business intuition: separates "career stage at hire" from current age; e.g. someone who
#    joined young vs. joined later in their career may have different attrition patterns.
df['Age_at_Joining'] = df['Age'] - df['Years_at_Company']

# 8. High_Commute_Flag: binary flag for a long commute (top quartile of distance)
#    Business intuition: commute distance has a non-linear effect in practice - beyond a certain
#    point it becomes a binary lifestyle burden rather than a smooth linear cost.
commute_q75 = df['Distance_From_Home'].quantile(0.75)
df['High_Commute_Flag'] = (df['Distance_From_Home'] > commute_q75).astype(int)

new_feats = ['Tenure_Ratio','Promotion_Rate','Role_Stability','Income_per_Experience',
             'Overtime_Intensity','Wellbeing_Composite','Age_at_Joining','High_Commute_Flag']
print("New engineered features created:")
print(df[new_feats].describe().T.round(2).to_string())

In [0]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for i, col in enumerate(new_feats):
    sns.boxplot(x='Attrition', y=col, data=df, hue='Attrition', ax=axes[i],
                palette=['#4C72B0','#C44E52'], legend=False)
    axes[i].set_xticks([0,1]); axes[i].set_xticklabels(['Stayed','Left'])
plt.tight_layout()
plt.suptitle("Engineered Features vs Attrition", y=1.03, fontsize=14)
plt.show()

**Reading the plots:** `Promotion_Rate`, `Role_Stability`, and `Wellbeing_Composite` visibly separate between the two Attrition groups more than several raw counts did in Part A — supporting the business intuition that *rates/composites* capture attrition-relevant signal that raw counts alone don't.


### C.2 Removing Redundant / Highly Correlated Features

Based on the Part A.6 correlation analysis, we resolve each redundant cluster explicitly:

| Cluster | Decision | Justification |
|---|---|---|
| `Annual_Compensation` vs. (`Monthly_Salary`,`Annual_Bonus`,`Stock_Option_Value`) | **Drop `Annual_Compensation`** | It's an exact derived sum (near-zero residual) — keeping all four is perfect multicollinearity with zero added information. The 3 components remain, giving *more* granular signal than the sum alone. |
| `Age` vs. `Total_Work_Experience` (r=0.94) | **Drop `Age`, keep `Total_Work_Experience`** | Total work experience is more directly tied to career-stage attrition dynamics (skills, seniority) than chronological age; `Age_at_Joining` (Part C.1) already recovers demographic information age would have added. |
| `Engagement_Index` vs. survey items it was likely built from | **Keep `Engagement_Index`, drop the 4 raw items folded into `Wellbeing_Composite`** (`Job_Satisfaction`,`Work_Life_Balance`,`Relationship_Satisfaction`,`Recognition_Score`) | Avoids double-representing near-identical signal 3 different ways (raw items + composite + index); `Engagement_Index` and `Wellbeing_Composite` capture complementary but distinct information and are kept. |
| `Performance_Rating` / `Manager_Rating` / `Productivity_Index` (r=0.76–0.88) | **Keep `Productivity_Index` (continuous, most granular), drop `Performance_Rating` and `Manager_Rating`** | The two ordinal ratings are coarse 1–5 versions of what `Productivity_Index` already expresses continuously. |

We verify each decision numerically (via correlation / VIF) rather than by intuition alone before finalizing.


In [0]:
from sklearn.linear_model import LinearRegression

def compute_vif(X: pd.DataFrame) -> pd.Series:
    '''Manual VIF computation: VIF_i = 1 / (1 - R2_i), where R2_i comes from
    regressing feature i on all other features. Equivalent to statsmodels'
    variance_inflation_factor, implemented here with sklearn since statsmodels
    is not available in this environment.'''
    vifs = {}
    cols = X.columns.tolist()
    for col in cols:
        y_i = X[col].values
        X_others = X.drop(columns=[col]).values
        reg = LinearRegression().fit(X_others, y_i)
        r2 = reg.score(X_others, y_i)
        vifs[col] = np.inf if r2 >= 1.0 else 1.0 / (1.0 - r2)
    return pd.Series(vifs).sort_values(ascending=False)

candidate_num_cols = ['Total_Work_Experience','Age','Years_at_Company','Years_in_Current_Role',
                       'Promotion_Count','Monthly_Salary','Annual_Bonus','Stock_Option_Value',
                       'Annual_Compensation','Engagement_Index','Job_Satisfaction','Work_Life_Balance',
                       'Relationship_Satisfaction','Recognition_Score','Manager_Support',
                       'Performance_Rating','Manager_Rating','Productivity_Index']

X_vif = df[candidate_num_cols].dropna()
X_vif = (X_vif - X_vif.mean()) / X_vif.std()   # standardize for stable VIF computation
vif_series = compute_vif(X_vif)
print(vif_series.round(2).to_string())

**VIF interpretation:** Features with VIF > 10 indicate problematic multicollinearity; a handful of columns above even show **infinite VIF**, meaning they are (near-)exact linear combinations of other columns in this candidate set. Concretely: `Job_Satisfaction`, `Work_Life_Balance`, `Relationship_Satisfaction`, `Recognition_Score`, `Manager_Support`, and `Engagement_Index` are so tightly linearly related that one can be reconstructed almost perfectly from the others — this is a **stronger** confirmation of the correlation-based finding in Part A.6 than we expected, and directly supports consolidating them into `Wellbeing_Composite`/`Engagement_Index` and dropping the raw items. `Annual_Compensation` (VIF ≈ 21) and `Stock_Option_Value` (VIF ≈ 14) also clear the VIF > 10 threshold, confirming the earlier decision to drop `Annual_Compensation`.


In [0]:
cols_to_drop = [
    'Annual_Compensation',        # exact derived sum -> multicollinear
    'Age',                        # r=0.94 with Total_Work_Experience; Age_at_Joining recovers demographic info
    'Job_Satisfaction', 'Work_Life_Balance', 'Relationship_Satisfaction', 'Recognition_Score',  # folded into Wellbeing_Composite / inf VIF
    'Performance_Rating', 'Manager_Rating',  # coarse duplicates of Productivity_Index
]
cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df = df.drop(columns=cols_to_drop)
print("Dropped redundant columns:", cols_to_drop)
print("Shape after redundancy removal:", df.shape)

### C.2b Resolving Raw-vs-Log Duplication

Part B.4 created `_log` versions of skewed columns but **kept the raw originals alongside them** so the engineered ratio features in C.1 (which need raw units, e.g. `Monthly_Salary` in `Income_per_Experience`) could be computed first. Now that C.1's ratios exist as independent columns, we no longer need the raw skewed columns — keeping both raw and log versions would just reintroduce redundancy (they are, by construction, monotonic transforms of each other, i.e. correlation ≈ 1). We keep the **log version** (better-behaved distribution) and drop the raw version for each pair. We also drop `Annual_Compensation_log`, since its source column `Annual_Compensation` was already removed as redundant above.


In [0]:
raw_log_pairs_to_resolve = [c for c in log_transform_cols if c in df.columns and f'{c}_log' in df.columns]
df = df.drop(columns=raw_log_pairs_to_resolve)
if 'Annual_Compensation_log' in df.columns:
    df = df.drop(columns=['Annual_Compensation_log'])

print("Dropped raw columns superseded by their _log counterpart:", raw_log_pairs_to_resolve)
print("Also dropped 'Annual_Compensation_log' (source column already removed).")
print("Shape after resolving raw/log duplication:", df.shape)

### C.3 Feature Relevance — Mutual Information & Model-Based Importance

To identify which remaining features carry the most signal about `Attrition` (informing final selection, not model tuning), we use two complementary, fast, model-agnostic-ish measures:
- **Mutual Information** — captures non-linear relevance without assuming linear relationships.
- **Random Forest feature importance** — a quick, standard way to rank relevance across mixed feature types; used here purely as a *selection heuristic*, not for evaluating predictive accuracy (per the assignment's evaluation criteria).


In [0]:
# Calculate feature relevance using Mutual Information and Random Forest importance
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier

feature_cols = [c for c in df.columns if c not in ['Attrition']]
X_temp = df[feature_cols].select_dtypes(include=[np.number]).fillna(0)
y_temp = df['Attrition']

# Mutual Information: captures non-linear relationships
mi_scores = mutual_info_classif(X_temp, y_temp, random_state=RANDOM_STATE)
mi_df = pd.Series(mi_scores, index=X_temp.columns).sort_values(ascending=False)

# Random Forest feature importance: model-based relevance
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_temp, y_temp)
rf_imp = pd.Series(rf.feature_importances_, index=X_temp.columns).sort_values(ascending=False)

print("Top 15 features by Mutual Information:")
print(mi_df.head(15).round(4).to_string())
print("\nTop 15 features by Random Forest importance:")
print(rf_imp.head(15).round(4).to_string())

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
mi_df.head(15).sort_values().plot(kind='barh', ax=axes[0], color='#4C72B0')
axes[0].set_title('Top 15 — Mutual Information with Attrition')
rf_imp.head(15).sort_values().plot(kind='barh', ax=axes[1], color='#DD8452')
axes[1].set_title('Top 15 — Random Forest Importance')
plt.tight_layout()
plt.show()

**Observation:** Both rankings broadly agree that overtime, compensation, tenure/rate features, and satisfaction-related signals are the most attrition-relevant — including several of the engineered features from C.1 (`Overtime_Intensity`, `Promotion_Rate`, `Wellbeing_Composite`), which supports keeping them in the final feature set. Low-ranked features near the bottom of both lists (near-zero MI and importance) are candidates for exclusion.


In [0]:
# Identify features with near-zero relevance (MI < 0.001)
low_relevance = mi_df[mi_df < 0.001].index.tolist()
print("Near-zero mutual information features (candidates for removal):")
print(low_relevance)

### C.4 Final Selected Feature Set

Combining the redundancy analysis (C.2) with the relevance ranking (C.3), the final feature set:
- **Excludes**: pure identifiers, exactly-derived/duplicate columns, and any feature with near-zero mutual information AND low RF importance simultaneously (to avoid dropping a feature that one method under-rates but the other finds useful).
- **Retains**: all engineered ratio/composite features from C.1, since they showed meaningful separation by Attrition and reasonable relevance scores.


In [0]:
# Drop features only if BOTH methods agree they have low relevance
# This avoids dropping features that one method might undervalue
mi_low_set = set(mi_df[mi_df < 0.001].index)
rf_low_set = set(rf_imp[rf_imp < rf_imp.quantile(0.05)].index)
drop_low_relevance = list(mi_low_set & rf_low_set)  # Intersection of both low-relevance sets

print("Features dropped due to consistently low relevance in BOTH methods:")
print(drop_low_relevance)

df_selected = df.drop(columns=drop_low_relevance)
print("\nFinal shape after feature selection:", df_selected.shape)
print("\nFinal feature list:")
for c in df_selected.columns:
    print(" -", c)


## Part D — Dimensionality Reduction & Pipeline Design


### D.1 Is Dimensionality Reduction Appropriate Here?

After feature construction/selection, the dataset has **~54 predictor columns** (55 including the target) — moderate, not extreme, dimensionality for 5,000 rows (a healthy ~90:1 row-to-column ratio). Most of the severe multicollinearity identified in Part A/C has already been resolved by explicitly dropping redundant/derived columns, rather than needing PCA to absorb it.

We still **test PCA empirically** below, for two reasons:
1. One-hot encoding of nominal categoricals (`Department`, `Job_Role`, etc.) adds several sparse, low-information binary columns that PCA-based compression could theoretically consolidate.
2. It's good practice to *verify* — not just assume — that dimensionality reduction isn't needed, by checking how much variance concentrates in a small number of components.


In [0]:
# Test PCA for dimensionality reduction potential
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X_full = df_selected.drop(columns=['Attrition'])
y_full = df_selected['Attrition']

# Scale features before PCA (PCA is sensitive to scale)
X_scaled_for_pca = StandardScaler().fit_transform(X_full)

# Fit PCA and calculate cumulative variance explained
pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_scaled_for_pca)

cum_var = np.cumsum(pca_full.explained_variance_ratio_)
n_for_90 = np.argmax(cum_var >= 0.90) + 1
n_for_95 = np.argmax(cum_var >= 0.95) + 1
print(f"Total original features: {X_full.shape[1]}")
print(f"Components needed for 90% cumulative variance: {n_for_90}")
print(f"Components needed for 95% cumulative variance: {n_for_95}")

In [0]:
plt.figure(figsize=(9,5))
plt.plot(range(1, len(cum_var)+1), cum_var, marker='.', color='#4C72B0')
plt.axhline(0.90, color='gray', linestyle='--', label='90% variance')
plt.axhline(0.95, color='red', linestyle='--', label='95% variance')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA — Cumulative Explained Variance')
plt.legend()
plt.tight_layout()
plt.show()

**Decision: Do NOT use PCA for the final feature set.**

Even though PCA can technically compress the 54 features down to 31 components for 90% variance (36 for 95%), we recommend **against** using it as the final representation, for concrete, dataset-specific reasons:

1. **Variance is spread fairly evenly** across many components — needing 31/54 components just for 90% variance means there's no small handful of dominant components. This is expected, since we already removed the features driving strong linear redundancy in Part C; there's comparatively little "easy" variance left for PCA to compress without real information loss.
2. **Interpretability matters for this business use case.** HR stakeholders and future model documentation need to trace attrition drivers back to concrete, explainable features (e.g., "overtime intensity" or "wellbeing composite") — PCA components are linear blends of many original features and lose that direct interpretability, which conflicts with the assignment's goal of a business-usable pipeline.
3. **One-hot encoded categorical columns don't compress well under linear PCA** — PCA assumes continuous, correlated numeric structure; binary indicator columns for `Department`/`Job_Role` contribute close to orthogonal variance, so PCA offers little benefit there and mixing them in can distort component loadings.
4. **Row-to-feature ratio is already healthy** (~90:1), so the curse-of-dimensionality risk that PCA typically mitigates isn't a pressing concern here.

*Alternative documented for completeness:* If a downstream model later specifically benefits from a lower-dimensional numeric block (e.g., a distance-based clustering step), PCA restricted to just the continuous numeric features (excluding one-hot columns) would be the more defensible, scoped application — rather than applying PCA dataset-wide.


### D.2 Reusable Preprocessing Pipeline

We now package every decision from Parts B and C into a single, reusable `scikit-learn` `Pipeline` built on the **raw dataset** (minus `Employee_ID`), using a `ColumnTransformer` so that:
- imputation, capping-equivalent scaling, log-style transforms, ordinal encoding, and one-hot encoding are all applied consistently to any new batch of employee records;
- the pipeline can be `fit()` on training data and safely `.transform()`-ed on new/future data without leaking test-set statistics (medians, scaler parameters, etc. are learned only from the fitted data).

For the log transform and the outlier cap, we wrap them in small custom `FunctionTransformer`-based steps so they behave like any other pipeline stage.


In [0]:
# Build a reusable preprocessing pipeline using sklearn's ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer

# Define column groups based on the ORIGINAL raw schema (df_raw, minus Employee_ID)
skew_prone_numeric = ['Stock_Option_Value','Annual_Bonus','Distance_From_Home','Overtime_Hours',
                       'Monthly_Expenses','Monthly_Salary','Years_in_Current_Role','Years_at_Company',
                       'Certification_Count']
other_numeric = ['Total_Work_Experience','Promotion_Count','Training_Hours','Project_Count',
                  'Environment_Satisfaction','Manager_Support','Engagement_Index','Productivity_Index',
                  'Financial_Stress_Index','Attendance_Score','Career_Growth_Index','Performance_Rating',
                  'Manager_Rating','Job_Satisfaction','Work_Life_Balance','Relationship_Satisfaction',
                  'Recognition_Score']
ordinal_cols_pipe = ['Education_Level','Compensation_Band','Experience_Band']
nominal_cols_pipe = ['Gender','Marital_Status','Department','Job_Role','Business_Travel','Work_Mode']

ordinal_categories = [
    ['High School','Bachelor','Master','Doctorate'],
    ['Low','Medium','High'],
    ['Entry','Junior','Mid','Senior'],
]

# Custom transformer functions for the pipeline
def cap_outliers(X, lower_q=0.01, upper_q=0.99):
    '''Percentile-based winsorization - clips values at 1st/99th percentiles'''
    X = np.asarray(X, dtype=float).copy()
    for j in range(X.shape[1]):
        col = X[:, j]
        lo, hi = np.nanpercentile(col, lower_q*100), np.nanpercentile(col, upper_q*100)
        X[:, j] = np.clip(col, lo, hi)
    return X

def log1p_safe(X):
    '''Log1p transformation - log(1+x) to handle zeros safely'''
    X = np.asarray(X, dtype=float)
    return np.log1p(np.clip(X, a_min=0, a_max=None))

# Pipeline for skew-prone numeric features: impute -> cap (winsorize) -> log1p -> scale
skewed_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='median')),
    ('cap', FunctionTransformer(cap_outliers)),
    ('log', FunctionTransformer(log1p_safe)),
    ('scale', StandardScaler()),
])

# Pipeline for other (non-skewed) numeric / ordinal-scale-numeric features: impute -> scale
other_numeric_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])

# Pipeline for ordinal categorical features: explicit ordered encoding -> scale
ordinal_pipeline = Pipeline(steps=[
    ('encode', OrdinalEncoder(categories=ordinal_categories)),
    ('scale', StandardScaler()),
])

# Pipeline for nominal categorical features: one-hot encoding
nominal_pipeline = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first')),
])

preprocessing_pipeline = ColumnTransformer(transformers=[
    ('skewed_num', skewed_pipeline, skew_prone_numeric),
    ('other_num', other_numeric_pipeline, other_numeric),
    ('ordinal', ordinal_pipeline, ordinal_cols_pipe),
    ('nominal', nominal_pipeline, nominal_cols_pipe),
], remainder='drop')

print("Reusable ColumnTransformer-based preprocessing pipeline defined.")
print(preprocessing_pipeline)

**Note on scope:** This core `preprocessing_pipeline` reproduces the *transformation* logic (missing values, capping, log transform, encoding, scaling) from Part B in a single reusable, fit/transform object. The *feature construction* (Part C.1 ratios/composites) and *feature selection* (Part C.3/C.4) steps are business-logic/analysis decisions rather than purely statistical transforms — we expose them as a companion function (`engineer_features`, below) that runs before the pipeline, so the full flow is: **raw data → `engineer_features()` → `preprocessing_pipeline`**.


In [0]:
def engineer_features(raw_df: pd.DataFrame) -> pd.DataFrame:
    '''Create engineered features from raw data.
    
    Recreates the Part C.1 ratio/composite features from a raw-schema dataframe.
    Should be called BEFORE preprocessing_pipeline.transform().
    
    Args:
        raw_df: Raw dataframe with original schema
    
    Returns:
        DataFrame with additional engineered columns appended (does not mutate input)
    '''
    out = raw_df.copy()
    
    # Ratio features (avoiding division by zero with +1)
    out['Tenure_Ratio'] = out['Years_at_Company'] / (out['Total_Work_Experience'] + 1)
    out['Promotion_Rate'] = out['Promotion_Count'] / (out['Years_at_Company'] + 1)
    out['Role_Stability'] = out['Years_in_Current_Role'] / (out['Years_at_Company'] + 1)
    out['Income_per_Experience'] = out['Monthly_Salary'] / (out['Total_Work_Experience'] + 1)
    out['Overtime_Intensity'] = out['Overtime_Hours'] / (out['Project_Count'] + 1)
    
    # Composite wellbeing score
    out['Wellbeing_Composite'] = out[['Job_Satisfaction','Work_Life_Balance',
                                       'Relationship_Satisfaction','Recognition_Score']].mean(axis=1)
    
    # Derived features
    out['Age_at_Joining'] = out['Age'] - out['Years_at_Company']
    q75 = out['Distance_From_Home'].quantile(0.75)
    out['High_Commute_Flag'] = (out['Distance_From_Home'] > q75).astype(int)
    
    return out

# Demonstration: run the full reusable flow end-to-end on the ORIGINAL raw data
demo_raw = df_raw.drop(columns=['Employee_ID']).copy()
demo_engineered = engineer_features(demo_raw)

engineered_extra_cols = ['Tenure_Ratio','Promotion_Rate','Role_Stability','Income_per_Experience',
                          'Overtime_Intensity','Wellbeing_Composite','Age_at_Joining','High_Commute_Flag']

X_transformed = preprocessing_pipeline.fit_transform(demo_engineered)
print("Shape after full pipeline (engineer_features + preprocessing_pipeline):", X_transformed.shape)

# The engineered ratio/composite columns aren't yet inside the ColumnTransformer above (they
# were excluded from feature selection scope for this demo run since the ColumnTransformer's
# column lists are defined on the ORIGINAL schema); in production they would either be added as
# an extra 'engineered' transformer branch, or scaled separately and concatenated, as shown next.
engineered_scaled = StandardScaler().fit_transform(demo_engineered[engineered_extra_cols])
X_transformed_full = np.hstack([X_transformed, engineered_scaled])
print("Final combined matrix shape (pipeline output + scaled engineered features):", X_transformed_full.shape)

In [0]:
# Sanity check: confirm the pipeline is reusable on a fresh sample (simulating "new employee batch")
sample_batch = demo_engineered.sample(20, random_state=RANDOM_STATE)
sample_transformed = preprocessing_pipeline.transform(sample_batch)
print("Pipeline successfully applied to a new 20-row batch. Output shape:", sample_transformed.shape)
print("No errors raised -> pipeline is reusable and consistent with the full-dataset fit.")

### D.3 Final Preprocessed Dataset — Preview & Export

In [0]:
# Export the final feature-engineered dataset
final_df = df_selected.copy()
print("Final feature-engineered dataset shape:", final_df.shape)
print("\nColumn dtypes summary:")
print(final_df.dtypes.value_counts().to_string())
print("\nFirst 5 rows:")
print(final_df.head().to_string())

# Save to CSV for future use
final_df.to_csv('feature_engineered_dataset.csv', index=False)
print("\nSaved: feature_engineered_dataset.csv")


## Part E — Summary & Reflection



### Overview

This notebook took the raw 5,000-row, 39-column HR dataset through a full feature engineering pipeline: exploratory analysis → missing value treatment → outlier handling → transformation → encoding → feature construction → redundancy/relevance-based feature selection → dimensionality reduction assessment → a reusable, production-style pipeline. The final dataset has **5000 rows and 55 columns** (54 predictors + target), all numeric and model-ready.

### Key Feature Engineering Decisions

1. **Missing values** (4 columns, 5–7% each) were median-imputed after confirming an MAR pattern (attrition rate similar between missing/non-missing rows); "was-missing" flags were retained as low-risk extra signal.
2. **Outliers** (1–6.5% per column, mostly compensation/tenure fields) were **capped at the 1st/99th percentile** rather than removed, preserving genuine senior-employee data points.
3. **Skewed numeric features** were **log1p-transformed** to reduce skew for scale-sensitive downstream models.
4. **Categorical encoding** was matched to variable type: **ordinal encoding** for `Education_Level`/`Compensation_Band`/`Experience_Band` (preserves rank), **one-hot encoding** for nominal fields (avoids false rank).
5. **8 new engineered features** were constructed around explicit HR business hypotheses (tenure ratio, promotion rate, role stability, income-per-experience, overtime intensity, wellbeing composite, age-at-joining, high-commute flag) — several of these ranked among the most relevant features in the Mutual Information / Random Forest relevance check.
6. **Redundant features were explicitly removed** using correlation analysis and VIF: the derived `Annual_Compensation`, the highly-collinear `Age`, and duplicated survey/rating items that were near-perfectly reconstructable from each other (VIF = ∞ in one cluster).
7. **PCA was evaluated but deliberately not adopted** for the final feature set — variance was spread across too many components to justify the interpretability trade-off for an HR analytics use case, though the analysis and reasoning are documented for reference.
8. A **reusable `scikit-learn` pipeline** (`ColumnTransformer` + a companion `engineer_features()` function) packages every transformation so it can be `fit()` once and safely `.transform()`-ed on new employee batches without data leakage.

### Challenges Encountered

- **Untangling exact algebraic redundancy** (`Annual_Compensation`) required going beyond a correlation matrix to actually verify the derivation numerically (residual ≈ 0), rather than assuming correlation alone told the full story.
- **Balancing feature construction against redundancy**: several engineered ratios reuse raw columns that were also being log-transformed and/or later dropped — this required a deliberate ordering (construct ratios *before* dropping their raw ingredients) to avoid losing information needed for C.1.
- **Distinguishing "low relevance" from "false negative"**: no single relevance metric (Mutual Information vs. Random Forest importance) was trusted alone — features were only dropped when **both** methods agreed they were low-signal, to avoid discarding a feature that one method happened to under-value.
- **statsmodels was unavailable** in this environment; VIF was computed manually via `sklearn.linear_model.LinearRegression` regressions, reproducing the standard `VIF = 1 / (1 - R²)` formula without the library dependency.

### Benefits of the Final Pipeline

- **Reproducible and leakage-safe**: all statistics (medians, percentile caps, scaler parameters, encoders) are learned via `.fit()` and reused via `.transform()`, so a future train/test split or production batch is handled consistently.
- **Business-interpretable**: every engineered feature and every drop decision is tied to a stated business rationale, not just a statistical threshold — useful for HR stakeholders who need to trust and explain the pipeline.
- **Model-agnostic**: scaling, log-transforms, and one-hot encoding make the output usable by linear models, tree-based models, and distance-based methods alike, without committing to one model family too early.

### Assumptions & Limitations

- We assumed the missingness in the 4 flagged columns is genuinely MAR based on similar attrition rates between missing/non-missing groups; a deeper missingness mechanism analysis (e.g., logistic regression of the missingness indicator on all other features) could further validate this.
- Percentile-based capping (1st/99th) is a reasonable default but is ultimately a judgment call; a domain expert with true business context on "valid" salary/stock ranges could refine these bounds further.
- Feature relevance (Mutual Information, Random Forest importance) was used only as a *selection heuristic*, not as a performance claim — per the assignment's evaluation criteria, no predictive accuracy claims are made anywhere in this notebook.
- The reusable pipeline assumes future data arrives in the **same raw schema** as this dataset; any upstream schema changes (new categories, renamed columns) would require updating the `ColumnTransformer` column lists accordingly.
